Step 0: Import config


In [2]:
from config import config as cf

Step 1: Load dataset

In [3]:
dataset = cf.pd.read_csv(cf.basic_dataset, sep=cf.separator)
print(dataset.dtypes)

Date                                                                             str
Service                                                                          str
Departure station                                                                str
Arrival station                                                                  str
Average journey time                                                             str
Number of scheduled trains                                                       str
Number of cancelled trains                                                       str
Cancellation comments                                                            str
Number of trains delayed at departure                                            str
Average delay of late trains at departure                                        str
Average delay of all trains at departure                                         str
Departure delay comments                                         

Step 3: Convert type of date to datetime

In [4]:
dataset[cf.date] = cf.pd.to_datetime(dataset[cf.date], format="%Y-%m", errors="coerce").dt.date
dataset.dropna(subset=[cf.date], inplace=True)
dataset[cf.date] = cf.pd.to_datetime(dataset[cf.date])
print(dataset.dtypes)


Date                                                                             datetime64[s]
Service                                                                                    str
Departure station                                                                          str
Arrival station                                                                            str
Average journey time                                                                       str
Number of scheduled trains                                                                 str
Number of cancelled trains                                                                 str
Cancellation comments                                                                      str
Number of trains delayed at departure                                                      str
Average delay of late trains at departure                                                  str
Average delay of all trains at departure          

Step 4: Convert columns types in numeric when necessary

In [5]:
dataset[cf.average_journey_time] = cf.pd.to_numeric(dataset[cf.average_journey_time], errors="coerce")
dataset[cf.number_train_sheduled] = cf.pd.to_numeric(dataset[cf.number_train_sheduled], errors="coerce")
dataset[cf.number_train_cancel] = cf.pd.to_numeric(dataset[cf.number_train_cancel], errors="coerce")
dataset[cf.number_train_delayed_departure] = cf.pd.to_numeric(dataset[cf.number_train_delayed_departure], errors="coerce")
dataset[cf.average_delay_late_train_at_departure] = cf.pd.to_numeric(dataset[cf.average_delay_late_train_at_departure], errors="coerce")
dataset[cf.average_delay_all_at_departure] = cf.pd.to_numeric(dataset[cf.average_delay_all_at_departure], errors="coerce")
dataset[cf.number_train_delayed_arrival] = cf.pd.to_numeric(dataset[cf.number_train_delayed_arrival], errors="coerce")
dataset[cf.average_delay_late_train_at_arrival] = cf.pd.to_numeric(dataset[cf.average_delay_late_train_at_arrival], errors="coerce")
dataset[cf.average_delay_all_train_at_arrival] = cf.pd.to_numeric(dataset[cf.average_delay_all_train_at_arrival], errors="coerce")
dataset[cf.number_train_delay_sup_15min] = cf.pd.to_numeric(dataset[cf.number_train_delay_sup_15min], errors="coerce")
dataset[cf.average_delay_sup_15min_competing_flight] = cf.pd.to_numeric(dataset[cf.average_delay_sup_15min_competing_flight], errors="coerce")
dataset[cf.number_train_delay_sup_30min] = cf.pd.to_numeric(dataset[cf.number_train_delay_sup_30min], errors="coerce")
dataset[cf.number_train_delay_sup_60min] = cf.pd.to_numeric(dataset[cf.number_train_delay_sup_60min], errors="coerce")
dataset[cf.pct_delay_external_cause] = cf.pd.to_numeric(dataset[cf.pct_delay_external_cause], errors="coerce")
dataset[cf.pct_delay_infrastructure] = cf.pd.to_numeric(dataset[cf.pct_delay_infrastructure], errors="coerce")
dataset[cf.pct_delay_traffic] = cf.pd.to_numeric(dataset[cf.pct_delay_traffic], errors="coerce")
dataset[cf.pct_rolling_stock] = cf.pd.to_numeric(dataset[cf.pct_rolling_stock], errors="coerce")
dataset[cf.pct_delay_traffic] = cf.pd.to_numeric(dataset[cf.pct_delay_traffic], errors="coerce")
dataset[cf.pct_equipement_station_management] = cf.pd.to_numeric(dataset[cf.pct_equipement_station_management], errors="coerce")
dataset[cf.pct_passenger] = cf.pd.to_numeric(dataset[cf.pct_passenger], errors="coerce")

print(dataset.dtypes)

Date                                                                             datetime64[s]
Service                                                                                    str
Departure station                                                                          str
Arrival station                                                                            str
Average journey time                                                                   float64
Number of scheduled trains                                                             float64
Number of cancelled trains                                                             float64
Cancellation comments                                                                      str
Number of trains delayed at departure                                                  float64
Average delay of late trains at departure                                              float64
Average delay of all trains at departure          

Step 6: Add new feature columns

In [7]:
dataset[cf.year] = dataset[cf.date].dt.year
dataset[cf.month] = dataset[cf.date].dt.month_name()

def get_delay_category(delay):
    if (delay < 15):
        return "Minimum delay"
    if (delay < 30):
        return "Low delay"
    if (delay < 60):
        return "Medium delay"
    return "Significant delay"

def is_go_to_Paris(arrival):
    return "PARIS" in str(arrival)

dataset[cf.delay_category] = dataset[cf.average_delay_all_train_at_arrival].apply(get_delay_category)
dataset[cf.rate_cancel_train] =  (dataset[cf.number_train_cancel] / dataset[cf.number_train_sheduled].replace(0, float('nan'))) * 100
dataset[cf.rate_delay_train] =  (dataset[cf.number_train_delayed_arrival] / dataset[cf.number_train_sheduled].replace(0, float('nan'))) * 100
dataset[cf.go_to_Paris] = dataset[cf.arrival].apply(is_go_to_Paris)
print(dataset.loc[dataset[cf.rate_delay_train].idxmax()])

Date                                                                             2019-09-01 00:00:00
Service                                                                                     National
Departure station                                                                   BORDEAUX ST JEAN
Arrival station                                                                            TOURCOING
Average journey time                                                                           301.0
Number of scheduled trains                                                                      20.0
Number of cancelled trains                                                                       0.0
Cancellation comments                                                                            NaN
Number of trains delayed at departure                                                           10.0
Average delay of late trains at departure                                                  